# 🔬 AI-Driven Pharmaceutical Market Intelligence & Demand Forecasting
**Project 5 | Global Health + Pharma Analytics**

---
This notebook covers the full analytical pipeline:
1. Data Loading & Cleaning
2. Exploratory Data Analysis (EDA)
3. Correlation Analysis
4. Predictive Forecasting (2024–2030)
5. Business Insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## STEP 1 — Data Loading & Cleaning

In [ ]:
# Load dataset
df = pd.read_csv('../data/global_health_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(10)

In [ ]:
# Data Cleaning
print('=== Data Quality Report ===')
print(f'Missing values:\n{df.isnull().sum()}')

# Clean
df = df.dropna()
df['Year'] = df['Year'].astype(int)
df['Drug_Sales_USD'] = pd.to_numeric(df['Drug_Sales_USD'], errors='coerce')
df['Prevalence_Pct'] = pd.to_numeric(df['Prevalence_Pct'], errors='coerce')
df = df[df['Drug_Sales_USD'] > 0]
df = df[df['Prevalence_Pct'] > 0]

print(f'\nCleaned dataset: {df.shape[0]} rows')
print(f'Countries: {df["Country"].nunique()}')
print(f'Diseases: {df["Disease"].nunique()}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')

## STEP 2 — Exploratory Data Analysis

In [ ]:
# Q1: Which diseases drive highest drug demand?
disease_sales = df.groupby('Disease')['Drug_Sales_USD'].sum().sort_values(ascending=False) / 1e9
print('=== Drug Sales by Disease (USD Billions) ===')
for disease, sales in disease_sales.items():
    print(f'{disease:<30} ${sales:.1f}B')

In [ ]:
# Q2: Which countries show fastest growth?
growth_data = df.groupby(['Country','Year'])['Drug_Sales_USD'].sum().reset_index()
growth_data = growth_data.sort_values(['Country','Year'])
growth_rates = []
for country in growth_data['Country'].unique():
    cdf = growth_data[growth_data['Country']==country]
    start = cdf[cdf['Year']==2015]['Drug_Sales_USD'].values
    end = cdf[cdf['Year']==2023]['Drug_Sales_USD'].values
    if len(start) > 0 and len(end) > 0:
        cagr = ((end[0]/start[0])**(1/8)-1)*100
        growth_rates.append({'Country':country,'CAGR_Pct':round(cagr,2),'Market_Type':cdf['Market_Type'].iloc[0]})

gr_df = pd.DataFrame(growth_rates).sort_values('CAGR_Pct',ascending=False)
print('=== Country CAGR Rankings (2015-2023) ===')
print(gr_df.to_string(index=False))

In [ ]:
# Q3: Correlation - Healthcare Spending vs Drug Demand
corr_df = df.groupby(['Country','Year']).agg(
    Drug_Sales_B=('Drug_Sales_USD','sum'),
    Avg_Health_Spend=('Healthcare_Spending_GDP_Pct','mean')
).reset_index()
corr_df['Drug_Sales_B'] = corr_df['Drug_Sales_B'] / 1e9
pearsons_r = corr_df['Avg_Health_Spend'].corr(corr_df['Drug_Sales_B'])
print(f'Pearson Correlation (Healthcare Spend vs Drug Sales): r = {pearsons_r:.4f}')
print(f'R-squared: {pearsons_r**2:.4f}')
print(f'Interpretation: {"Strong positive" if pearsons_r > 0.6 else "Moderate positive" if pearsons_r > 0.3 else "Weak"} correlation')

## STEP 3 — Predictive Forecasting (2024–2030)

In [ ]:
# Polynomial regression forecasting per disease
from sklearn.preprocessing import PolynomialFeatures

forecasts = {}
future_years = np.array(range(2024, 2031)).reshape(-1,1)

for disease in sorted(df['Disease'].unique()):
    ddf = df[df['Disease']==disease].groupby('Year')['Drug_Sales_USD'].sum().reset_index()
    X = ddf['Year'].values.reshape(-1,1)
    y = ddf['Drug_Sales_USD'].values / 1e9
    poly = PolynomialFeatures(degree=2)
    model = LinearRegression()
    model.fit(poly.fit_transform(X), y)
    y_future = model.predict(poly.transform(future_years))
    r2 = r2_score(y, model.predict(poly.transform(X)))
    forecasts[disease] = {'2025': round(y_future[1],2), '2030': round(y_future[-1],2), 'R2': round(r2,3)}

print('=== FORECAST RESULTS (USD Billions) ===')
print(f'{"Disease":<30} {"2025 Est.":>12} {"2030 Est.":>12} {"Model R²":>10}')
print('-'*65)
for d, f in forecasts.items():
    print(f'{d:<30} ${f["2025"]:>10.1f}B ${f["2030"]:>10.1f}B {f["R2"]:>10.3f}')

## STEP 4 — Summary Statistics

In [ ]:
print('=== COMPREHENSIVE MARKET SUMMARY ===')
total_sales = df['Drug_Sales_USD'].sum() / 1e12
print(f'Total Market Value (2015-2023): ${total_sales:.2f} Trillion')

print(f'\nTop 3 Markets by Total Revenue:')
top3 = df.groupby('Country')['Drug_Sales_USD'].sum().sort_values(ascending=False).head(3) / 1e9
for c, v in top3.items():
    print(f'  {c}: ${v:.1f}B')

print(f'\nFastest Growing Disease Market (by CAGR):')
disease_growth = []
for disease in df['Disease'].unique():
    ddf = df[df['Disease']==disease].groupby('Year')['Drug_Sales_USD'].sum()
    cagr = ((ddf.iloc[-1]/ddf.iloc[0])**(1/8)-1)*100
    disease_growth.append({'Disease':disease,'CAGR':round(cagr,1)})
dg = pd.DataFrame(disease_growth).sort_values('CAGR',ascending=False)
for _, row in dg.head(3).iterrows():
    print(f'  {row["Disease"]}: {row["CAGR"]}% CAGR')
    
print(f'\nEmerging vs Established Markets:')
market_split = df.groupby('Market_Type')['Drug_Sales_USD'].sum() / 1e9
for mt, val in market_split.items():
    pct = val / market_split.sum() * 100
    print(f'  {mt}: ${val:.1f}B ({pct:.1f}% of total)')